In [1]:
import os

In [2]:
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/YohanJay23/wine_quality_end_to_end_project.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "YohanJay23"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "12aa8bae310f0795c7c23dcd43e1bb74fec38ea1"

In [3]:
%pwd

'c:\\Users\\2021ICTS28\\Desktop\\wine_quality_end_to_end_project\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\2021ICTS28\\Desktop\\wine_quality_end_to_end_project'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    metrics_file_path: Path
    all_params: dict
    target_column: str
    mlflow_uri: str
 

In [7]:
from src.wineProject.utils.common import read_yaml, create_directories, save_json
from src.wineProject.constants import *

In [13]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path = CONFIG_FILE_PATH,
        params_file_path = PARAMS_FILE_PATH,
        schema_file_path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)
        
        create_directories([self.config.artifacts_root])
        
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN
        
        create_directories([config.root_dir])
        
        model_evaluation_config = ModelEvaluationConfig(
            root_dir = config.root_dir,
            test_data_path = config.test_data_path,
            model_path = config.model_path,
            metrics_file_path = config.metrics_file_path,
            all_params = params,
            target_column = schema.name,  
            mlflow_uri = "https://dagshub.com/YohanJay23/wine_quality_end_to_end_project.mlflow"
        )
        
        return model_evaluation_config

In [14]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [17]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
        
    def eval_metrics(self, actual, predicted) -> tuple:
        rmse = np.sqrt(mean_squared_error(actual, predicted))
        mae = mean_absolute_error(actual, predicted)
        r2 = r2_score(actual, predicted)
        return rmse, mae, r2
    
    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)
        
        test_X = test_data.drop(self.config.target_column, axis=1)
        test_y = test_data[[self.config.target_column]]
        
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            predicted_qualities = model.predict(test_X)
            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)
            
            scores ={   
                "rmse": rmse,
                "mae": mae,
                "r2": r2
            }
            
            save_json(path=Path(self.config.metrics_file_path), data=scores)
            mlflow.log_params(self.config.all_params)
            
            mlflow.log_metrics(scores)
            
            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model" ,registered_model_name="ElasticNetWineModel")
            else:
                mlflow.sklearn.log_model(model, "model")

In [18]:
try:
    config = ConfigurationManager()
    evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e
    
    

[2026-03-21 11:40:43,093: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-03-21 11:40:43,096: INFO: common]: yaml file: params.yaml loaded successfully
[2026-03-21 11:40:43,097: INFO: common]: yaml file: schema.yaml loaded successfully
[2026-03-21 11:40:43,098: INFO: common]: created directory at: artifacts
[2026-03-21 11:40:43,099: INFO: common]: created directory at: artifacts/model_evaluation
[2026-03-21 11:40:45,137: INFO: common]: json file saved at: artifacts\model_evaluation\metrics.json


2026/03/21 11:40:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/21 11:40:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'ElasticNetWineModel'.
2026/03/21 11:40:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticNetWineModel, version 1
Created version '1' of model 'ElasticNetWineModel'.


🏃 View run thundering-horse-947 at: https://dagshub.com/YohanJay23/wine_quality_end_to_end_project.mlflow/#/experiments/0/runs/db72fbd4695b427dba19c6c7d12af3dd
🧪 View experiment at: https://dagshub.com/YohanJay23/wine_quality_end_to_end_project.mlflow/#/experiments/0
